# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [2]:
import os

FEATURES = ["P_dyn", "VBs", "epsilon", "DST"]
MODE = "template"  # 'template' or 'default'
OUTPUT_DIR = "comparison_template_deriv_against_baselines_newv"
RAW_EQ = "g = (#2 * -0.0010713526) * sqrt(#1 + 1.32442); d = square((#1 * 0.01838707) - 0.5912093)"

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/mnt/data/symbolic-regression-dst-public-repo/.venv/lib/python3.12/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()


In [4]:
raw_data = load_and_preprocess(save_prepared_data=True)
data = compute_features(raw_data)

Reading from file ./data/all_timeline/ace_imf_1h_1998.csv
Reading from file ./data/all_timeline/ace_imf_1h_1999.csv
Reading from file ./data/all_timeline/ace_imf_1h_2000.csv
Reading from file ./data/all_timeline/ace_imf_1h_2001.csv
Reading from file ./data/all_timeline/ace_imf_1h_2002.csv
Reading from file ./data/all_timeline/ace_imf_1h_2003.csv
Reading from file ./data/all_timeline/ace_imf_1h_2004.csv
Reading from file ./data/all_timeline/ace_imf_1h_2005.csv
Reading from file ./data/all_timeline/ace_imf_1h_2006.csv
Reading from file ./data/all_timeline/ace_imf_1h_2007.csv
Reading from file ./data/all_timeline/ace_imf_1h_2008.csv
Reading from file ./data/all_timeline/ace_imf_1h_2009.csv
Reading from file ./data/all_timeline/ace_imf_1h_2010.csv
Reading from file ./data/all_timeline/ace_imf_1h_2011.csv
Reading from file ./data/all_timeline/ace_imf_1h_2012.csv
Reading from file ./data/all_timeline/ace_imf_1h_2013.csv
Reading from file ./data/all_timeline/ace_imf_1h_2014.csv
Reading from f

In [5]:
def predict_and_plot_storm(model, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values

    res_eq = simulate_storm(model, storm_df)
    res_burton = baseline_models.burton_prediction(storm_df)
    res_obm = baseline_models.obm_prediction(storm_df)
    ddm1 = baseline_models.ddm1_prediction(storm_df)
    ddm2 = baseline_models.ddm2_prediction(storm_df)
    ddm3 = baseline_models.ddm3_prediction(storm_df)

    res_eq = res_eq[start:end]["DST_pred"].values
    res_burton = res_burton[start:end]["DST_pred"].values
    res_obm = res_obm[start:end]["DST_pred"].values
    res_ddm1 = ddm1[start:end]["DST_pred"].values
    res_ddm2 = ddm2[start:end]["DST_pred"].values
    res_ddm3 = ddm3[start:end]["DST_pred"].values

    # 2. Calculate Metrics
    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 8), constrained_layout=True)
    fig.suptitle(
        rf"Evaluation for Equation: ${model.latex_str()}$", fontsize=18, wrap=True
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_eq,
        color="blue",
        linestyle="--",
        label="Equation",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_burton,
        color="yellow",
        linestyle="--",
        label="Burton",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_obm,
        color="green",
        linestyle="--",
        label="OBM",
        linewidth=1.5,
    )
    
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm1,
        color="orange",
        linestyle="--",
        label="DDM1",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm2,
        color="purple",
        linestyle="--",
        label="DDM2",
        linewidth=1.5,
    )
    axs[0].plot(
        storm_df[start:end].index,
        res_ddm3,
        color="cyan",
        linestyle="--",
        label="DDM3",
        linewidth=1.5,
    )
    
    axs[0].set_title(f"Storm {storm_id} Reconstruction", fontsize=18)
    axs[0].tick_params(axis='both', which='major', labelsize=14)
    axs[0].tick_params(axis='both', which='minor', labelsize=10)
    axs[0].legend(fontsize = 16)
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    axs[0].set_xlabel("Date", fontsize=16)
    axs[0].set_ylabel("Dst (nT)", fontsize=16)
    axs[0].xaxis.set_major_locator(MultipleLocator(2))

    # Column 2: Prediction Error
    diff_eq = res_eq - y_true
    diff_burton = res_burton - y_true
    diff_obm = res_obm - y_true
    diff_ddm1 = res_ddm1 - y_true
    diff_ddm2 = res_ddm2 - y_true
    diff_ddm3 = res_ddm3 - y_true


    axs[1].plot(storm_df[start:end].index, diff_eq, color="blue", label="Eq Error")
    axs[1].plot(
        storm_df[start:end].index, diff_burton, color="yellow", label="Burton Error"
    )
    axs[1].plot(storm_df[start:end].index, diff_obm, color="green", label="OBM Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm1, color="orange", label="DDM1 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm2, color="purple", label="DDM2 Error")
    axs[1].plot(storm_df[start:end].index, diff_ddm3, color="cyan", label="DDM3 Error")
    axs[1].axhline(0, color="black", linestyle="--")
    axs[1].set_title("Prediction Error", fontsize=18)
    axs[1].set_ylabel("Error (nT)", fontsize=16)
    axs[1].set_xlabel("Date", fontsize=16)

    title_metrics = (
        f"Error Comparison\n"
        f"Eq: RMSE {m_eq[0]:.2f} | MAE {m_eq[1]:.2f} | R2 {m_eq[2]:.2f} | CC {m_eq[3]:.2f} | BFE {m_eq[4]:.2f}\n"
        f"Burton: RMSE {m_burton[0]:.2f} | MAE {m_burton[1]:.2f} | R2 {m_burton[2]:.2f} | CC {m_burton[3]:.2f} | BFE {m_burton[4]:.2f}\n"
        f"OBM: RMSE {m_obm[0]:.2f} | MAE {m_obm[1]:.2f} | R2 {m_obm[2]:.2f} | CC {m_obm[3]:.2f} | BFE {m_obm[4]:.2f}\n"
        f"DDM1: RMSE {m_ddm1[0]:.2f} | MAE {m_ddm1[1]:.2f} | R2 {m_ddm1[2]:.2f} | CC {m_ddm1[3]:.2f} | BFE {m_ddm1[4]:.2f}\n"
        f"DDM2: RMSE {m_ddm2[0]:.2f} | MAE {m_ddm2[1]:.2f} | R2 {m_ddm2[2]:.2f} | CC {m_ddm2[3]:.2f} | BFE {m_ddm2[4]:.2f}\n"
        f"DDM3: RMSE {m_ddm3[0]:.2f} | MAE {m_ddm3[1]:.2f} | R2 {m_ddm3[2]:.2f} | CC {m_ddm3[3]:.2f} | BFE {m_ddm3[4]:.2f}"
    )
    axs[1].set_title(title_metrics, fontsize=18)
    axs[1].set_ylabel("Error (nT)", fontsize=16)
    axs[1].set_xlabel("Date", fontsize=16)
    axs[1].legend(fontsize=16)
    axs[1].grid(True)
    axs[1].set_xlim(start, end)
    axs[1].tick_params(axis='both', which='major', labelsize=14)
    axs[1].tick_params(axis='both', which='minor', labelsize=10)
    
    axs[1].xaxis.set_major_locator(MultipleLocator(2))

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        [res_eq, res_burton, res_obm, res_ddm1, res_ddm2, res_ddm3],
        ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"],
        ["blue", "yellow", "green", "orange", "purple", "cyan"],
        fontsize = 16   
    )

    plt.savefig(save_path)
    plt.close()


In [6]:
def save_prediction_data(model, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eq = simulate_storm(model, storm_df)
    if model.is_template:
        pred_dst_eq = pred_dst_eq[start:end][
            ["DST_pred", "dDST", "injection_component", "decay_component"]
        ]
    else:
        pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
    # 3. Baseline Predictions (Burton & OBM)
    pred_dst_burton = baseline_models.burton_prediction(storm_df)
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred", "dDST"]]
    pred_dst_burton.columns = ["DST_pred_burton", "dDST_burton"]
    pred_dst_burton = pred_dst_burton[start:end][["DST_pred_burton", "dDST_burton"]]
    pred_dst_obm = baseline_models.obm_prediction(storm_df)
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred", "dDST"]]
    pred_dst_obm.columns = ["DST_pred_obm", "dDST_obm"]
    pred_dst_obm = pred_dst_obm[start:end][["DST_pred_obm", "dDST_obm"]]
    pred_dst_ddm1 = baseline_models.ddm1_prediction(storm_df)    
    pred_dst_ddm1.columns = ["DST_pred_ddm1", "dDST_ddm1"]
    pred_dst_ddm1 = pred_dst_ddm1[start:end][["DST_pred_ddm1", "dDST_ddm1"]]
    pred_dst_ddm2 = baseline_models.ddm2_prediction(storm_df)
    pred_dst_ddm2.columns = ["DST_pred_ddm2", "dDST_ddm2"]
    pred_dst_ddm2 = pred_dst_ddm2[start:end][["DST_pred_ddm2", "dDST_ddm2"]]
    pred_dst_ddm3 = baseline_models.ddm3_prediction(storm_df)
    pred_dst_ddm3.columns = ["DST_pred_ddm3", "dDST_ddm3"]
    pred_dst_ddm3 = pred_dst_ddm3[start:end][["DST_pred_ddm3", "dDST_ddm3"]]


    # 4. Construct Comprehensive DataFrame

    if model.is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,
            }
        ).set_index("Timestamp")
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Pred_DST_Burton": pred_dst_burton["DST_pred_burton"].values,
                "Pred_dDST_dt_Burton": pred_dst_burton["dDST_burton"].values,
                "Pred_DST_OBM": pred_dst_obm["DST_pred_obm"].values,
                "Pred_dDST_dt_OBM": pred_dst_obm["dDST_obm"].values,
                "Pred_DST_DDM1": pred_dst_ddm1["DST_pred_ddm1"].values,
                "Pred_dDST_dt_DDM1": pred_dst_ddm1["dDST_ddm1"].values,
                "Pred_DST_DDM2": pred_dst_ddm2["DST_pred_ddm2"].values,
                "Pred_dDST_dt_DDM2": pred_dst_ddm2["dDST_ddm2"].values,
                "Pred_DST_DDM3": pred_dst_ddm3["DST_pred_ddm3"].values,
                "Pred_dDST_dt_DDM3": pred_dst_ddm3["dDST_ddm3"].values,

            }
        ).set_index("Timestamp")

    results_df.to_csv(output_path)
    return results_df

## Test storms

In [7]:
storms = []

model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION
storm_indices = []
for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'w') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(model.latex_str())}\n')

100%|██████████| 20/20 [00:25<00:00,  1.28s/it]


In [8]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    summary_df.loc[len(summary_df)] = [
        storm_indices[storm_index],
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq)
m_burton = baseline_models.get_all_metrics(y_true, res_burton)
m_obm = baseline_models.get_all_metrics(y_true, res_obm)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

summary_df.loc[len(summary_df)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df)

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,...,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
0,54.0,9.082450,7.150428,0.796405,0.922839,11.620567,22.273257,19.295634,-0.224415,0.825122,...,17.034353,14.718489,0.283836,0.819670,19.285046,17.429228,14.902320,0.250248,0.849179,20.944809
1,55.0,17.429458,13.783357,0.748936,0.886955,19.558619,22.561217,19.634219,0.579330,0.889945,...,20.994874,17.672795,0.635713,0.870809,22.088199,22.442717,18.173790,0.583737,0.864456,25.386413
2,56.0,13.276276,10.962537,0.613082,0.889956,15.586261,14.594609,9.904945,0.532425,0.864593,...,11.710892,9.329760,0.698945,0.874185,13.104050,11.378825,9.160091,0.715776,0.872556,12.828221
3,57.0,9.579700,7.665862,0.766690,0.899931,12.124595,18.949308,14.323111,0.087114,0.643703,...,8.303250,6.470139,0.824723,0.917542,11.441876,8.670891,6.332414,0.808857,0.904447,13.883947
4,58.0,9.165543,6.390252,0.797016,0.927846,15.584912,9.491523,7.361750,0.782321,0.889916,...,11.990044,9.586551,0.652635,0.926882,19.031650,11.862568,8.934847,0.659982,0.920503,19.970899
5,59.0,12.207318,10.241290,0.849938,0.955928,9.432197,15.884176,12.114917,0.745927,0.899436,...,10.840170,8.734074,0.881668,0.955437,13.497036,12.354960,9.578556,0.846286,0.954712,16.759531
6,60.0,17.617883,11.828793,0.777722,0.934481,26.807645,25.327401,19.852983,0.540622,0.930660,...,13.320289,10.330457,0.872938,0.963738,17.936865,14.717364,11.742180,0.844887,0.954268,19.772733
7,61.0,10.679534,8.252146,0.932006,0.968053,14.553100,40.396238,21.193252,0.027141,0.886214,...,22.490189,16.826801,0.698454,0.883757,30.616512,20.468681,15.214925,0.750226,0.886201,30.187252
8,62.0,11.394979,9.122644,0.698127,0.886644,12.820999,14.256121,10.561750,0.527503,0.821907,...,10.405858,7.936314,0.748260,0.888742,14.058279,11.805307,9.352887,0.675995,0.859124,16.881751
9,63.0,14.975924,12.167919,0.837054,0.944592,18.193426,22.695026,18.805391,0.625787,0.882456,...,20.014049,16.071049,0.708977,0.932586,28.117249,22.085126,17.377477,0.645630,0.929705,32.083276


In [9]:
display(summary_df.iloc[-2:, :].style.format({col: "{:.2f}" for col in columns}))

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,Burton_BFE,OBM_RMSE,OBM_MAE,OBM_R2,OBM_CC,OBM_BFE,DDM1_RMSE,DDM1_MAE,DDM1_R2,DDM1_CC,DDM1_BFE,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
20,Mean,13.82,10.56,0.79,0.92,16.38,26.00,18.22,0.43,0.83,29.05,16.77,13.05,0.72,0.92,17.88,18.75,14.59,0.66,0.89,23.89,18.20,14.04,0.69,0.90,22.84,18.66,14.10,0.68,0.89,24.26
21,Global,14.01,10.25,0.89,0.94,28.79,28.57,16.73,0.52,0.86,78.35,17.18,12.64,0.83,0.92,43.36,19.86,13.52,0.77,0.89,38.89,19.06,13.16,0.79,0.90,45.26,19.55,13.36,0.78,0.89,46.38


In [10]:
display(summary_df[['Storm Index', 'Equation_BFE', 'Burton_BFE', 'OBM_BFE', 'DDM1_BFE', 'DDM2_BFE', 'DDM3_BFE']])

,Storm Index,Equation_BFE,Burton_BFE,OBM_BFE,DDM1_BFE,DDM2_BFE,DDM3_BFE
0,54.0,11.620567,22.737069,15.339845,21.129540,19.285046,20.944809
1,55.0,19.558619,19.121577,16.826395,23.435284,22.088199,25.386413
2,56.0,15.586261,20.446094,17.798724,13.923395,13.104050,12.828221
3,57.0,12.124595,15.869737,10.829970,13.120449,11.441876,13.883947
4,58.0,15.584912,11.644470,12.822532,20.108684,19.031650,19.970899
5,59.0,9.432197,16.203182,9.970305,14.982619,13.497036,16.759531
6,60.0,26.807645,33.244838,15.577698,36.468144,17.936865,19.772733
7,61.0,14.553100,59.335394,23.901355,28.911696,30.616512,30.187252
8,62.0,12.820999,12.923521,13.610126,14.844234,14.058279,16.881751
9,63.0,18.193426,24.673526,21.192304,29.948383,28.117249,32.083276


In [11]:
print(summary_df.loc[summary_df["Storm Index"].isin([57, 68, 'Mean'])].to_latex(index=False, float_format="%.2f"))

\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
Storm Index & Equation_RMSE & Equation_MAE & Equation_R2 & Equation_CC & Equation_BFE & Burton_RMSE & Burton_MAE & Burton_R2 & Burton_CC & Burton_BFE & OBM_RMSE & OBM_MAE & OBM_R2 & OBM_CC & OBM_BFE & DDM1_RMSE & DDM1_MAE & DDM1_R2 & DDM1_CC & DDM1_BFE & DDM2_RMSE & DDM2_MAE & DDM2_R2 & DDM2_CC & DDM2_BFE & DDM3_RMSE & DDM3_MAE & DDM3_R2 & DDM3_CC & DDM3_BFE \\
\midrule
57.00 & 9.58 & 7.67 & 0.77 & 0.90 & 12.12 & 18.95 & 14.32 & 0.09 & 0.64 & 15.87 & 10.28 & 8.56 & 0.73 & 0.86 & 10.83 & 8.77 & 6.75 & 0.80 & 0.92 & 13.12 & 8.30 & 6.47 & 0.82 & 0.92 & 11.44 & 8.67 & 6.33 & 0.81 & 0.90 & 13.88 \\
68.00 & 27.92 & 20.66 & 0.95 & 0.98 & 30.01 & 71.85 & 52.86 & 0.64 & 0.92 & 75.14 & 43.64 & 32.15 & 0.87 & 0.98 & 49.73 & 26.23 & 19.68 & 0.95 & 0.98 & 29.76 & 41.57 & 29.79 & 0.88 & 0.95 & 48.47 & 50.34 & 34.75 & 0.82 & 0.96 & 54.65 \\
Mean & 13.82 & 10.56 & 0.79 & 0.92 & 16.38 & 26.00 & 18.22 & 0.43 & 0.83 & 29.05 & 16.77 & 13.05 & 0.72 

## Train storms

In [12]:
storms = []

train_storms = storm_dates.TRAIN_STORMS_SYMBOLIC_REGRESSION
storm_indices = []
for sd, ed, storm_id in tqdm(train_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        print(f'Storm {storm_id} has no data from {start} to {end}. Skipping.')
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        model,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            model, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)

100%|██████████| 53/53 [01:21<00:00,  1.54s/it]


In [13]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = ["Equation", "Burton", "OBM", "DDM1", "DDM2", "DDM3"]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    res_eq = storm["Pred_DST_Equation"].values
    res_burton = storm["Pred_DST_Burton"].values
    res_obm = storm["Pred_DST_OBM"].values
    res_ddm1 = storm["Pred_DST_DDM1"].values
    res_ddm2 = storm["Pred_DST_DDM2"].values
    res_ddm3 = storm["Pred_DST_DDM3"].values

    m_eq = baseline_models.get_all_metrics(y_true, res_eq)
    m_burton = baseline_models.get_all_metrics(y_true, res_burton)
    m_obm = baseline_models.get_all_metrics(y_true, res_obm)
    m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
    m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
    m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

    summary_df.loc[len(summary_df)] = [
        storm_indices[storm_index],
        *m_eq,
        *m_burton,
        *m_obm,
        *m_ddm1,
        *m_ddm2,
        *m_ddm3,
    ]

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values
res_eq = global_data["Pred_DST_Equation"].values
res_burton = global_data["Pred_DST_Burton"].values
res_obm = global_data["Pred_DST_OBM"].values
res_ddm1 = global_data["Pred_DST_DDM1"].values
res_ddm2 = global_data["Pred_DST_DDM2"].values
res_ddm3 = global_data["Pred_DST_DDM3"].values

m_eq = baseline_models.get_all_metrics(y_true, res_eq)
m_burton = baseline_models.get_all_metrics(y_true, res_burton)
m_obm = baseline_models.get_all_metrics(y_true, res_obm)
m_ddm1 = baseline_models.get_all_metrics(y_true, res_ddm1)
m_ddm2 = baseline_models.get_all_metrics(y_true, res_ddm2)
m_ddm3 = baseline_models.get_all_metrics(y_true, res_ddm3)

summary_df.loc[len(summary_df)] = [
    "Global",
    *m_eq,
    *m_burton,
    *m_obm,
    *m_ddm1,
    *m_ddm2,
    *m_ddm3,
]

display(summary_df)

,Storm Index,Equation_RMSE,Equation_MAE,Equation_R2,Equation_CC,Equation_BFE,Burton_RMSE,Burton_MAE,Burton_R2,Burton_CC,...,DDM2_RMSE,DDM2_MAE,DDM2_R2,DDM2_CC,DDM2_BFE,DDM3_RMSE,DDM3_MAE,DDM3_R2,DDM3_CC,DDM3_BFE
0,1.0,20.213585,17.117328,0.640361,0.871804,16.822423,29.156901,25.538353,0.251721,0.839101,...,22.568521,19.475600,0.551681,0.896125,14.829997,23.252721,19.882235,0.524086,0.889651,19.966095
1,2.0,22.835546,17.477713,-0.337692,0.578832,19.405386,12.625578,10.846565,0.591082,0.844957,...,18.874640,14.104114,0.086117,0.588612,18.701587,21.678056,16.169453,-0.205519,0.501767,21.927198
2,3.0,11.215986,8.793026,0.879061,0.949642,8.793067,21.695670,17.060808,0.547482,0.844832,...,14.543500,12.109214,0.796658,0.933442,19.227867,15.653912,12.848885,0.764421,0.924793,21.447613
3,4.0,17.213986,14.144183,0.778240,0.920668,19.953851,41.408010,29.564523,-0.283185,0.864301,...,20.949735,17.795732,0.671544,0.887207,21.903819,20.551589,17.276006,0.683910,0.878793,22.014097
4,5.0,18.665306,14.714571,0.766902,0.907633,24.272886,32.233862,22.613058,0.304827,0.899751,...,22.171150,16.574360,0.671115,0.855711,29.435293,22.597948,16.987666,0.658331,0.843171,29.757253
5,6.0,15.532397,12.371153,0.645368,0.883474,16.965733,27.830107,19.744801,-0.138494,0.879934,...,18.066086,15.496299,0.520234,0.820916,17.225424,17.936579,15.021498,0.527088,0.822802,17.003694
6,7.0,10.465127,8.609895,0.808674,0.906307,7.050362,19.666746,16.300287,0.324306,0.815489,...,12.238735,10.768601,0.738327,0.915610,14.937157,13.896036,12.057634,0.662661,0.897081,18.509792
7,8.0,12.263987,9.014927,0.847011,0.962559,13.263640,19.523686,15.070868,0.612279,0.934798,...,10.379121,8.237262,0.890424,0.959182,8.695989,11.590607,9.317260,0.863351,0.957484,10.319187
8,9.0,12.576511,9.406606,0.692773,0.918247,11.592131,17.192414,14.828763,0.425867,0.884437,...,11.275480,8.936104,0.753050,0.884648,10.333607,12.823987,9.634174,0.680563,0.852321,11.559362
9,10.0,13.050190,9.008404,0.639154,0.884235,16.813556,13.699719,11.276735,0.602341,0.862175,...,14.126297,8.265998,0.577191,0.811954,21.448999,16.740966,8.877678,0.406188,0.721946,25.359561


In [14]:
display(summary_df[['Storm Index', 'Equation_BFE', 'Burton_BFE', 'OBM_BFE', 'DDM1_BFE', 'DDM2_BFE', 'DDM3_BFE']])

,Storm Index,Equation_BFE,Burton_BFE,OBM_BFE,DDM1_BFE,DDM2_BFE,DDM3_BFE
0,1.0,16.822423,24.722894,10.111580,13.437801,14.829997,19.966095
1,2.0,19.405386,10.861256,12.884873,19.256149,18.701587,21.927198
2,3.0,8.793067,25.950905,17.414149,20.804152,19.227867,21.447613
3,4.0,19.953851,40.213707,18.849678,21.296023,21.903819,22.014097
4,5.0,24.272886,51.929814,23.564759,27.490935,29.435293,29.757253
5,6.0,16.965733,33.556980,14.085708,12.948621,17.225424,17.003694
6,7.0,7.050362,24.195663,13.020184,17.712507,14.937157,18.509792
7,8.0,13.263640,21.390040,10.522216,9.832557,8.695989,10.319187
8,9.0,11.592131,13.250031,7.515882,11.832959,10.333607,11.559362
9,10.0,16.813556,11.885662,12.972115,19.511731,21.448999,25.359561
